# K-Means Clustering: From First Principles

## Why Do We Need Clustering?

Many datasets contain measurements but no target label that tells us the group for each row. A retailer may have purchase frequency, average order value, and product-category counts, but no reliable customer-type column. A clustering method looks for structure in those measurements and assigns similar rows to the same group.

Clustering is exploratory: the algorithm returns a partition of the data, not a discovered truth about what the groups mean. A person must inspect the resulting clusters, check whether they are useful and stable, and give them an interpretation. K-Means is a widely used starting point because its objective and update steps are simple, but its geometric assumptions matter.

## Supervised vs Unsupervised Learning

In supervised learning, each training row includes features and a known target. A classifier learns to predict a category; a regressor learns to predict a number. Performance can be compared with held-out target values.

In unsupervised learning, there is no target column to teach the algorithm which answer is correct. K-Means receives only feature vectors and groups them according to distance. Its cluster numbers are arbitrary labels: cluster 0 is not inherently better, earlier, or more important than cluster 1.

Because there may be no ground-truth labels, evaluating clustering requires different questions: are the groups compact and separated under a chosen distance, do they remain similar under reasonable changes, and are they useful for the application?

## What Is K-Means?

K-Means partitions $n$ observations into a chosen number $K$ of nonempty clusters. Each observation is represented by a vector of $d$ numeric features, and each cluster is represented by its centroid, the coordinate-wise mean of its assigned observations.

The algorithm alternates between assigning each observation to its nearest centroid and recomputing the centroids from those assignments. Its name comes from the use of $K$ clusters and the mean as the cluster representative.

A centroid is generally not an observed row. The labels are also not semantic: K-Means may return cluster 0 and cluster 1, but it does not know that a group represents "frequent customers" or "at-risk accounts." Those meanings must be assessed after fitting.

## The Core Idea and Objective

K-Means needs a value of $K$ before it starts. It represents each cluster by a centroid, written $\boldsymbol{\mu}_k$. For each observation, it chooses the closest centroid and tries to keep the total squared distance to the assigned centroids small.

For data points $\mathbf{x}_1,\ldots,\mathbf{x}_n$, the objective is:

$$
J = \sum_{i=1}^{n} \min_{k\in\{1,\ldots,K\}} \left\|\mathbf{x}_i-\boldsymbol{\mu}_k\right\|_2^2
$$

The norm $\|\cdot\|_2$ is Euclidean distance. Squaring the distance means points farther from their centroid contribute disproportionately more to the objective. The cluster assignments and centroids are both unknown, so K-Means improves them in alternating steps:

1. **Assignment:** with centroids fixed, assign each point to its nearest centroid.
2. **Update:** with assignments fixed, replace each centroid by the mean of the points assigned to it.

The mean minimizes the sum of squared Euclidean distances within a cluster. Each step therefore cannot increase $J$. K-Means stops when assignments no longer change, centroid movement is sufficiently small, or an iteration limit is reached. The objective can end at a local solution, so the result can depend on the starting centroids.

## Distance and Feature Scaling

For two points in two dimensions, Euclidean distance is:

$$
d(\mathbf{x},\mathbf{z}) = \sqrt{(x_1-z_1)^2+(x_2-z_2)^2}
$$

For $d$ features, the squared Euclidean distance is:

$$
d^2(\mathbf{x},\mathbf{z}) = \sum_{j=1}^{d}(x_j-z_j)^2
$$

K-Means assigns points using this geometry and minimizes squared distances. A feature measured in thousands can dominate one measured between 0 and 1, even if both are equally meaningful. Standardizing features or scaling them to comparable ranges can prevent units alone from controlling the clusters.

Scaling is a modeling choice, not a cosmetic step: it changes distances and therefore can change the result. Fit any scaling transformation using only the training data when evaluating on held-out observations, then apply that same transformation to the held-out data.

## Assignment Step

With the centroids fixed, calculate each point's distance to each centroid and assign it to the nearest one:

$$
r_i = \underset{k\in\{1,\ldots,K\}}{\operatorname{argmin}}\;\left\|\mathbf{x}_i-\boldsymbol{\mu}_k\right\|_2^2
$$

Here, $r_i$ is the cluster assignment for point $i$. Squared distances can be used instead of distances because squaring preserves which nonnegative distance is smallest. The assignment step gives each point exactly one cluster; equal-distance ties are resolved by the implementation's tie rule.

## Update Step

With the assignments fixed, the centroid for cluster $k$ is the mean of the points assigned to it:

$$
\boldsymbol{\mu}_k = \frac{1}{n_k}\sum_{i:r_i=k}\mathbf{x}_i
$$

Here, $n_k$ is the number of points assigned to cluster $k$. The mean is not an arbitrary choice: it is the point that minimizes the sum of squared Euclidean distances to the observations in that cluster. The mean is calculated independently for every feature, so a centroid in two dimensions is the average $x$ coordinate paired with the average $y$ coordinate.

If a cluster receives no points, its mean is undefined. Implementations handle this empty-cluster case using a documented policy, such as reinitializing its centroid.

## Initialization, Convergence, and Local Solutions

K-Means begins with initial centroid positions, then repeats assignment and update. It stops when assignments are unchanged, centroid movement is below a tolerance, or a maximum number of iterations is reached. A stop condition means the updates have stabilized; it does not prove that the algorithm found the best possible partition.

The objective cannot increase during an exact assignment step or an exact mean update. However, the objective is not generally convex with respect to all assignments and centroids together, so different starting points can lead to different local solutions.

Random initialization can place several centroids in one region or leave a useful region without a starting centroid. K-Means++ chooses initial centers to spread them out probabilistically and is a common default. In practice, fitting multiple initializations and retaining the run with the lowest inertia reduces dependence on one unlucky start. A fixed random seed makes runs reproducible.

## Inertia

Inertia is the value of the K-Means objective: the sum of squared distances from each observation to its assigned centroid.

$$
\mathrm{Inertia} = \sum_{i=1}^{n}\left\|\mathbf{x}_i-\boldsymbol{\mu}_{r_i}\right\|_2^2
$$

Lower inertia means points are, on average in the squared-distance sense, closer to their assigned centroids. It is nonnegative and becomes 0 if every point is its own centroid. For a fixed dataset and feature representation, the best achievable inertia cannot increase as $K$ grows, because an added centroid can preserve the previous partition or improve it.

This means inertia alone does not identify the right number of clusters: choosing $K=n$ can make it zero but is usually not useful. Compare the reduction in inertia as $K$ changes, and consider separation, stability, and whether the groups answer a meaningful question.

## Choosing $K$ and the Elbow Method

$K$ controls the granularity of the partition. A small $K$ can merge distinct groups; a large $K$ can divide a coherent group into fragments. There is not always one objectively correct value because the answer depends on the data representation and the purpose of clustering.

The Elbow Method fits models for several values of $K$ and plots each model's inertia. Inertia falls as $K$ increases, but the improvement often becomes smaller. A bend in the curve can suggest a useful trade-off between fit and complexity. The bend may be gradual or ambiguous, so it is a guide rather than a formal test.

The Silhouette Coefficient is another useful diagnostic because it compares how close a point is to its own cluster with how close it is to a neighboring cluster. It is covered separately in the next topic. Also consider domain requirements, cluster sizes, stability across initializations or samples, and whether the resulting groups are interpretable.

## Worked Inertia Comparison Across $K$

Consider one-dimensional points $\{0,1,4,5\}$. For each candidate $K$, suppose K-Means finds these compact partitions. Inertia is the sum of squared distances from each point to its assigned centroid.

| $K$ | Clusters and centroids | Inertia |
|---:|---|---:|
| 1 | $\{0,1,4,5\}$, centroid $2.5$ | $(0-2.5)^2+(1-2.5)^2+(4-2.5)^2+(5-2.5)^2=17$ |
| 2 | $\{0,1\}$ at $0.5$; $\{4,5\}$ at $4.5$ | $0.25+0.25+0.25+0.25=1$ |
| 3 | $\{0\}$; $\{1\}$; $\{4,5\}$ at $4.5$ | $0.5$ |
| 4 | Each point is its own centroid | $0$ |

The largest reduction is from $K=1$ to $K=2$ (17 to 1); adding a third or fourth cluster reduces inertia by only another $0.5$ each. The elbow therefore suggests $K=2$ as a compact summary, though the best choice still depends on the use case and other diagnostics. In practice, compare the best result across multiple initializations for each $K$.

## Worked Example: One Assignment and Update

Consider four two-dimensional points and choose $K=2$:

| Point | Coordinates |
|---|---|
| A | $(1,1)$ |
| B | $(2,1)$ |
| C | $(8,8)$ |
| D | $(9,8)$ |

Start with centroids $\boldsymbol{\mu}_1=(1,1)$ and $\boldsymbol{\mu}_2=(8,8)$. Compare squared Euclidean distances; taking square roots is unnecessary because it would not change the nearest centroid.

| Point | To $\boldsymbol{\mu}_1$ | To $\boldsymbol{\mu}_2$ | Assignment |
|---|---:|---:|---|
| A | $0$ | $98$ | Cluster 1 |
| B | $1$ | $85$ | Cluster 1 |
| C | $98$ | $0$ | Cluster 2 |
| D | $113$ | $1$ | Cluster 2 |

Now update each centroid by averaging the coordinates in its cluster:

$$
\boldsymbol{\mu}_1 = \left(\frac{1+2}{2},\frac{1+1}{2}\right)=(1.5,1),
\qquad
\boldsymbol{\mu}_2 = \left(\frac{8+9}{2},\frac{8+8}{2}\right)=(8.5,8).
$$

The assignments stay the same on the next pass, so the algorithm has reached a stable partition. The new inertia is $4(0.5^2)=1$: each point is 0.5 units horizontally from its centroid. Before the update, inertia was $0+1+0+1=2$. This concrete decrease illustrates how moving each centroid to its cluster mean improves the objective.

## Strengths, Limitations, and Evaluation

K-Means is relatively simple to implement and scale to many observations. Its objective is clear, and the centroids provide a concise summary of each cluster. It is a useful baseline when numeric features and distance-based groups are appropriate.

Its limitations follow from that objective:

- $K$ must be chosen before fitting, and results may depend on initialization.
- Euclidean distance is sensitive to feature scales, so preprocessing can change the result substantially.
- Means and squared distances are sensitive to outliers.
- The method favors compact, roughly round clusters; it can split curved groups or merge groups with very different densities or sizes.
- Every observation is assigned to a cluster, even if it is far from all centroids; basic K-Means has no dedicated noise label.
- In high-dimensional spaces, distance may become less informative, making clusters harder to distinguish.

For an unsupervised evaluation, inspect inertia together with measures such as the Silhouette Coefficient, cluster sizes, stability under different initializations or samples, and domain usefulness. If meaningful reference labels are available, external measures can compare the partition with those labels, but those labels are not used to fit K-Means.

Lloyd's common implementation costs approximately $O(n K d I)$, where $n$ is the number of observations, $K$ the number of clusters, $d$ the number of features, and $I$ the number of iterations. Actual runtime also depends on initialization, implementation, and data structure.

Use K-Means when compact, distance-based groups are plausible and a fixed number of groups is useful. Consider another method, such as DBSCAN or hierarchical clustering, when clusters may have irregular shapes, substantial noise, or a number of groups that should not be specified in advance.